In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
# Extraction
def extract_bronze_erp_cust_az12(spark, table: str = "`databricks-project`.bronze.erp_cust_az12") -> DataFrame:
    """Load source data from the bronze layer."""
    print(f">> Reading from: {table}")
    return spark.table(table)

In [0]:
# Transformation Helpers
def clean_cid_erp_cust_az12(col_name: str) -> F.Column:
    """
    Remove 'NAS' prefix from cid if present.
    Mirrors:
        CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
             ELSE cid
        END
    """
    col = F.col(col_name)
    return (
        F.when(col.startswith("NAS"), F.substring(col, 4, F.length(col)))
         .otherwise(col)
    ).alias(col_name)


def clean_bdate_erp_cust_az12(col_name: str) -> F.Column:
    """
    Set future birthdates to NULL.
    Mirrors:
        CASE WHEN bdate > GETDATE() THEN NULL
             ELSE bdate
        END
    """
    col = F.col(col_name)
    return (
        F.when(col > F.current_date(), F.lit(None))
         .otherwise(col)
    ).alias(col_name)


def normalize_gender_erp_cust_az12(col_name: str) -> F.Column:
    """
    Normalize gender values to 'Female', 'Male', or 'n/a'.
    Mirrors:
        CASE WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
             WHEN UPPER(TRIM(gen)) IN ('M', 'MALE')   THEN 'Male'
             ELSE 'n/a'
        END
    """
    cleaned = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(cleaned.isin("F", "FEMALE"), "Female")
         .when(cleaned.isin("M", "MALE"),   "Male")
         .otherwise("n/a")
    ).alias(col_name)

In [0]:
# Transformation
def transform_erp_cust_az12(df: DataFrame) -> DataFrame:
    """
    Apply all cleaning and normalization rules to produce
    the silver-layer version of erp_cust_az12.
    """
    df_clean = df.select(
        clean_cid_erp_cust_az12("cid").alias("customer_number"),
        clean_bdate_erp_cust_az12("bdate").alias("birth_date"),
        normalize_gender_erp_cust_az12("gen").alias("gender"),
    )

    return df_clean

In [0]:
# Loading
def load_silver_erp_cust_az12(df: DataFrame, table: str = "`databricks-project`.silver.erp_cust_az12") -> None:
    """
    Overwrite the silver target table.
    Uses 'overwrite' mode to replicate TRUNCATE + INSERT behaviour.
    """
    print(f">> Truncating & inserting into: {table}")
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable(table)
    )
    print(f">> Load complete: {table}")

In [0]:
# Pipeline Entry Point
def run_pipeline_erp_cust_az12(
    source_table: str = "`databricks-project`.bronze.erp_cust_az12",
    target_table: str = "`databricks-project`.silver.erp_cust_az12",
) -> None:
    """End-to-end ETL pipeline for silver.erp_cust_az12."""

    bronze_df = extract_bronze_erp_cust_az12(spark, source_table)
    silver_df = transform_erp_cust_az12(bronze_df)
    load_silver_erp_cust_az12(silver_df, target_table)

    print(">> Pipeline finished successfully.")